# 06 · Entraînement du modèle d'attrition — MLflow + Optuna + Feature Store
1. Jeu d'entraînement construit par le Feature Store (jointure point-in-time labels ↔ features).
2. Découpage temporel : train (2025-07 → 2025-12), tuning (2026-01 → 02), test (2026-03 → 04).
3. Baseline logistique, puis Optuna sur un gradient boosting (maximise la PR-AUC : ~3 % de partants).
4. Modèle final entraîné sur train + tuning, évalué une fois sur le test, enregistré avec l'alias `challenger`.

In [0]:
%pip install -q databricks-feature-engineering optuna
%restart_python

In [0]:
import os, sys
sys.path.append(os.path.abspath("../src"))
import mlflow, optuna
import pandas as pd
from mlflow.tracking import MlflowClient
from databricks.feature_engineering import FeatureEngineeringClient, FeatureLookup
from churn.dbx import get_config, table_version
from churn.config import FEATURES, LABEL, TEST_SNAPSHOTS, TRAIN_SNAPSHOTS, VALID_SNAPSHOTS, in_window
from churn import modeling as M

cfg = get_config(dbutils)
fe = FeatureEngineeringClient()
mlflow.set_registry_uri("databricks-uc")
user = spark.sql("SELECT current_user()").first()[0]
mlflow.set_experiment(f"/Users/{user}/attrition-banque-modele")

dbutils.widgets.text("n_trials", "10")
n_trials = int(dbutils.widgets.get("n_trials"))
print("Modèle :", cfg.churn_model_name, "| essais Optuna :", n_trials)

In [0]:
ft_name = cfg.t("client_features")
labels = spark.table(cfg.t("attrition_labels"))          # client_id, snapshot_date, churn

training_set = fe.create_training_set(
    df=labels,
    feature_lookups=[FeatureLookup(table_name=ft_name,
                                   lookup_key="client_id",
                                   timestamp_lookup_key="snapshot_date")],   # jointure point-in-time
    label=LABEL)
df = training_set.load_df().toPandas()
df["snapshot_date"] = df["snapshot_date"].dt.date   # aligne le type sur date (attendu par in_window)

train = df[in_window(df.snapshot_date, TRAIN_SNAPSHOTS)]
valid = df[in_window(df.snapshot_date, VALID_SNAPSHOTS)]
test = df[in_window(df.snapshot_date, TEST_SNAPSHOTS)]
for name, part in [("train", train), ("tuning", valid), ("test", test)]:
    print(f"{name:7s}: {len(part):6d} lignes | taux d'attrition {part[LABEL].mean():.3f}")

In [0]:
def fit_eval(model_type, params, fit_df, eval_df):
    """Entraîne sur fit_df, évalue sur eval_df, renvoie (modèle, métriques)."""
    pipe = M.build_churn_pipeline(model_type, **params).fit(fit_df[FEATURES], fit_df[LABEL])
    return pipe, M.evaluate(eval_df[LABEL], pipe.predict_proba(eval_df[FEATURES])[:, 1])


with mlflow.start_run(run_name="attrition_training") as parent:
    # Traçabilité : quelles données exactement ont servi ? (versions Delta -> time travel)
    mlflow.log_params({
        "feature_table": ft_name,
        "feature_table_version": table_version(spark, ft_name),
        "labels_version": table_version(spark, cfg.t("attrition_labels")),
        "train_window": "→".join(TRAIN_SNAPSHOTS),
        "valid_window": "→".join(VALID_SNAPSHOTS),
        "test_window": "→".join(TEST_SNAPSHOTS),
        "horizon_jours": cfg.churn_horizon_days})

    # 1) Baseline interprétable
    with mlflow.start_run(run_name="baseline_logreg", nested=True):
        _, m = fit_eval("logreg", {}, train, valid)
        mlflow.log_metrics({f"valid_{k}": v for k, v in m.items()})
        print("Baseline logistique (tuning) :", {k: round(v, 3) for k, v in m.items()})

    # 2) Optuna : chaque essai = un run MLflow enfant
    def objective(trial):
        params = {
            "learning_rate": trial.suggest_float("learning_rate", 0.02, 0.3, log=True),
            "max_depth": trial.suggest_int("max_depth", 3, 10),
            "max_leaf_nodes": trial.suggest_int("max_leaf_nodes", 15, 63),
            "min_samples_leaf": trial.suggest_int("min_samples_leaf", 20, 200),
            "l2_regularization": trial.suggest_float("l2_regularization", 1e-4, 10, log=True),
        }
        with mlflow.start_run(run_name=f"hgb_trial_{trial.number}", nested=True):
            _, m = fit_eval("hgb", params, train, valid)
            mlflow.log_params(params)
            mlflow.log_metrics({f"valid_{k}": v for k, v in m.items()})
        return m["pr_auc"]                      # on maximise la PR-AUC (classes déséquilibrées)

    study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(objective, n_trials=n_trials)
    best = study.best_params
    mlflow.log_params({f"best_{k}": v for k, v in best.items()})
    mlflow.log_metric("best_valid_pr_auc", study.best_value)
    print("Meilleurs hyperparamètres :", best)

    # 3) Modèle final : entraîné sur train + tuning, évalué UNE fois sur le test
    final_pipe, test_m = fit_eval("hgb", best, pd.concat([train, valid]), test)
    mlflow.log_metrics({f"test_{k}": v for k, v in test_m.items()})
    print("Modèle final (test) :", {k: round(v, 3) for k, v in test_m.items()})

    # 4) Explicabilité globale : importance par permutation
    from sklearn.inspection import permutation_importance
    pi = permutation_importance(final_pipe, test[FEATURES], test[LABEL], scoring="roc_auc",
                                n_repeats=5, random_state=42)
    imp = (pd.DataFrame({"feature": FEATURES, "importance": pi.importances_mean})
           .sort_values("importance", ascending=False))
    mlflow.log_table(imp, "permutation_importance.json")

    # 5) Enregistrement via le Feature Store (lignage + lookup automatique au scoring)
    fe.log_model(
        model=M.ChurnProbaModel(final_pipe),     # renvoie une probabilité, pas 0/1
        artifact_path="model",
        flavor=mlflow.pyfunc,
        training_set=training_set,
        registered_model_name=cfg.churn_model_name,
        infer_input_example=True,
        code_paths=["../src/churn"])
    run_id = parent.info.run_id

In [0]:
client = MlflowClient()
version = max(int(v.version) for v in client.search_model_versions(f"name='{cfg.churn_model_name}'")
              if v.run_id == run_id)
client.set_registered_model_alias(cfg.churn_model_name, "challenger", version)
client.set_model_version_tag(cfg.churn_model_name, str(version), "test_pr_auc", f"{test_m['pr_auc']:.4f}")
print(f"Version {version} de {cfg.churn_model_name} → alias 'challenger'")
display(imp)